<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# Notebook 05: Downstream Detection with Foundation Model Embeddings

Three XGBoost models, same rows, same labels, different feature sets:

| Model | Features | Question it answers |
|---|---|---|
| **Baseline** | 16 raw columns | how far do the raw fields get you |
| **Embeddings** | 64d PCA of the 512d embeddings | does the sequence model alone carry the signal |
| **Combined** | raw + 64d PCA embeddings | do embeddings add anything the raw fields miss |

Rows come straight from notebook 04's aligned outputs -- embeddings, labels and raw
features were written in one pass, in one order, so there is no seed to reproduce and
no index to line up.

**Average Precision is the metric that matters.** At a low positive rate, AUROC
saturates and hides differences; AP measures precision where the review budget is spent.

In [ ]:
import json
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import cudf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import OrdinalEncoder

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.experiment import shared_data_dir
DATA_DIR = shared_data_dir()          # set in configs/paths.yaml (the only place)
SPLIT_DIR = DATA_DIR / "temporal_split"
EMBED_DIR = DATA_DIR / "embeddings"
OUTPUTS_DIR = DATA_DIR / "outputs"
TOKENIZER_DIR = DATA_DIR / "tokenizer"
# Run by scripts/run_experiment.py: every path comes from the run folder.
# Interactive use (LTM_EXPERIMENT unset): EXP is None and nothing changes.
from src.experiment import active_experiment
EXP = active_experiment()
if EXP is not None:
    globals().update(EXP.paths())
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

assert (EMBED_DIR / "metadata.json").exists(), (
    f"No embeddings in {EMBED_DIR}  (run notebook 04 first)"
)

XGB_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_STATE = 42
PCA_DIM = 64
LOGREG = True   # also fit logistic regression on each feature set (sanity check)
if EXP is not None:
    globals().update(EXP.xgboost_settings())   # PCA_DIM, RANDOM_STATE

metadata = json.loads((EMBED_DIR / "metadata.json").read_text())
print("Embedding metadata:")
for key, value in metadata.items():
    if not isinstance(value, (dict, list)):
        print(f"  {key}: {value}")
print(f"  splits: {metadata['splits']}")

## 1. Load Embeddings, Labels and Aligned Raw Features

In [ ]:
t0 = time.time()
data = {}
for split in ("train", "val", "test"):
    emb = np.load(EMBED_DIR / f"{split}_embeddings.npy")
    lbl = np.load(EMBED_DIR / f"{split}_labels.npy")
    feats = cudf.read_parquet(str(EMBED_DIR / f"{split}_features.parquet")).to_pandas()
    assert len(emb) == len(lbl) == len(feats), f"{split}: row count mismatch"
    assert (feats["_target"].values == lbl).all(), (
        f"{split}: saved features and labels disagree -- re-run notebook 04"
    )
    data[split] = {"emb": emb, "y": lbl.astype(int), "raw": feats}
print(f"Loaded in {time.time() - t0:.1f}s")

# Learned pooling: the attention head was fitted on some train customers, whose
# embeddings are therefore optimistic. Every arm -- raw included -- trains on the
# other customers only, so the comparison stays like for like.
if metadata.get("head_rows_file"):
    head_rows = np.load(EMBED_DIR / metadata["head_rows_file"])
    assert len(head_rows) == len(data["train"]["y"]), "head rows do not match train"
    keep = ~head_rows
    data["train"] = {
        "emb": data["train"]["emb"][keep],
        "y": data["train"]["y"][keep],
        "raw": data["train"]["raw"][keep].reset_index(drop=True),
    }
    print(f"Learned pooling: {int(head_rows.sum()):,} train rows fitted the head "
          f"and are left out; XGBoost trains on {int(keep.sum()):,}")

embed_dim = data["train"]["emb"].shape[1]
print(f"\n{'Split':<8} {'Rows':>10} {'Positives':>11} {'Rate':>9}")
print("-" * 42)
for split, d in data.items():
    print(f"{split:<8} {len(d['y']):>10,} {int(d['y'].sum()):>11,} {d['y'].mean():>8.4%}")
print(f"\nEmbedding dimension: {embed_dim} (history window: "
      f"{metadata['history_window']} transactions)")
print(f"Embedding stats (train): mean={data['train']['emb'].mean():.6f}, "
      f"std={data['train']['emb'].std():.6f}")

## 2. Dimensionality of the Embedding Arm

`USE_PCA = False` feeds XGBoost all 512 embedding dimensions. PCA keeps directions of
maximum **variance**, which need not be the directions carrying fraud signal -- an
unsupervised reduction before a supervised task can discard exactly what you need. At
1M x 512 float32 (2 GB) there is no compute reason to reduce, so the default is not to.

Set `USE_PCA = True` to report the 64d PCA arm instead; running both and reporting the
pair is a legitimate ablation.

In [ ]:
USE_PCA = False        # True restores the 64d PCA arm
if EXP is not None:
    USE_PCA = EXP.xgboost_settings().get("USE_PCA", USE_PCA)

if USE_PCA:
    t0 = time.time()
    pca = PCA(n_components=PCA_DIM, random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(data["train"]["emb"])
    X_val_pca = pca.transform(data["val"]["emb"])
    X_test_pca = pca.transform(data["test"]["emb"])
    EMB_LABEL = f"Embeddings ({PCA_DIM}d PCA of {embed_dim}d)"
    print(f"PCA {embed_dim}d -> {PCA_DIM}d in {time.time() - t0:.1f}s")
    print(f"  Explained variance: {pca.explained_variance_ratio_.sum():.2%}")
    print(f"  Top-10 components:  {pca.explained_variance_ratio_[:10].round(4).tolist()}")
else:
    X_train_pca = data["train"]["emb"]
    X_val_pca = data["val"]["emb"]
    X_test_pca = data["test"]["emb"]
    EMB_LABEL = f"Embeddings ({embed_dim}d, no PCA)"
    print(f"PCA disabled -- using all {embed_dim} embedding dimensions")

# All three are set together: a half-updated triple gives XGBoost mismatched
# eval_set shapes ("Invalid ref DMatrix, different number of features").
print(f"  train {X_train_pca.shape} | val {X_val_pca.shape} | test {X_test_pca.shape}")

## 3. Raw-Feature Matrix

Three departures from the upstream encoding, all about **generalising under a temporal
split**. The diagnostic that motivated them: a clearly higher in-sample AUC than val/test, which
were near chance -- the model could fit the labels, but nothing transferred.

1. **`customer_id` dropped.** Ordinal-encoded over ~10^4 levels it was the single most
   important feature, and all it can express is *which customers offended during
   training*. The tokenizer excludes it for exactly this reason: it is
   the sequence key, not a feature.

2. **`_ts` dropped.** Test timestamps lie entirely outside the training range -- 0%
   overlap, by construction, because the split is temporal. Trees cannot extrapolate, so
   every test row falls past the last split point. Measured: test AUC no better with it than
   without (both near chance). *Relative* time is fine and is what the tokenizer uses (`TDIF` bins the
   gap between transactions); absolute time is not.

3. **High-cardinality identifiers frequency-encoded, not ordinal-encoded.** `cp_account`
   (~10^5 levels) is genuinely fraud-relevant -- but an ordinal code is an arbitrary
   sort position, and roughly a tenth of test values were unseen and collapsed to `-1`. What
   generalises is not *which* account but how often it is seen and whether it is new, so
   each becomes two columns: `log1p(train frequency)` and an `unseen` flag. Novelty is
   itself a fraud signal. This mirrors the tokenizer, where ~10^5 accounts hash into 4096
   buckets: a bucket cannot identify an account, only whether it recurs in a customer's
   history -- so both arms now work from the same generalisable information.

Low-cardinality categoricals are still ordinal-encoded, but listed **explicitly** rather
than selected by dtype. That also removes the `make_column_selector(dtype_include=[...])`
hazard, where a pandas `string`-dtype column holding a `"MISSING"`
sentinel is matched by neither `"object"` nor `"category"`.

In [ ]:
# NB05 deliberately does not import the tokenizer (that would pull in cuDF), so
# column names come from the schema notebook 01 writes beside the tokenizer state.
SCHEMA_PATH = TOKENIZER_DIR / "schema.json"
assert SCHEMA_PATH.exists(), f"Missing {SCHEMA_PATH}  (written by notebook 01)"
SCH = json.loads(SCHEMA_PATH.read_text())

present = list(data["train"]["raw"].columns)


def _pick(*fields):
    """Schema field names -> the columns actually present in the frame."""
    out = []
    for field in fields:
        col = SCH.get(field)
        if col and col in present and col not in out:
            out.append(col)
    return out


# The tokenizer config notebook 02 used: its drop_fields and extra_fields decide
# what the raw arm sees, so both arms get the same information.
_STATE = TOKENIZER_DIR / "tokenizer_state.json"
TOK_CFG = json.loads(_STATE.read_text())["metadata"]["config"] if _STATE.exists() else {}
# Token prefix -> schema field, for the built-in fields.
PREFIX_FIELD = {
    "EMP": "employee", "BIC": "counterparty_bic", "CTRY": "counterparty_country",
    "CMT": "comment_structure", "OPST": "operations_status", "MVC": "movement_code",
    "SIGN": "movement_sign", "DC": "dc_flag", "CPTY": "counterparty_account",
    "AMT": "amount", "HOUR": "hour", "DOW": "dow", "MONTH": "month",
}
DROPPED = [p for p in (TOK_CFG.get("drop_fields") or [])]

# Dropped entirely: pure identity, absolute time, and the fields notebook 02 drops
# from the tokens (CONFIG.drop_fields), so both arms see the same information.
DROP_COLS = _pick("customer_id", "datetime", *[PREFIX_FIELD[p] for p in DROPPED if p in PREFIX_FIELD])

# Frequency encoded (high cardinality, nominal).
HIGH_CARD = _pick("counterparty_account", "account_number", "counterparty_bic")

# Ordinal encoded (low cardinality, genuinely categorical).
CAT_COLS = _pick("counterparty_country", "employee", "comment_structure",
                 "operations_status", "movement_code", "movement_sign", "dc_flag")

# Passed through as numbers.
NUM_COLS = _pick("amount", "month", "dow", "hour")

# Fields added through extra_fields, encoded by their kind. Text is not encoded
# for the raw arm yet -- it shows up in the NOTE below.
for _name, _spec in (TOK_CFG.get("extra_fields") or {}).items():
    _col = _spec["column"]
    if _name in DROPPED or _col not in present:
        if _name in DROPPED and _col in present:
            DROP_COLS.append(_col)
        continue
    {"categorical": CAT_COLS, "identifier": HIGH_CARD, "numeric": NUM_COLS}.get(
        _spec["kind"], []).append(_col)
# The relative amount the tokens carry (include_relative_amount), as a number.
if TOK_CFG.get("include_relative_amount") and "amt_rel" in present:
    NUM_COLS.append("amt_rel")
CAT_COLS = [c for c in CAT_COLS if c not in DROP_COLS]
HIGH_CARD = [c for c in HIGH_CARD if c not in DROP_COLS]
NUM_COLS = [c for c in NUM_COLS if c not in DROP_COLS]

unassigned = [c for c in present
              if c not in DROP_COLS + HIGH_CARD + CAT_COLS + NUM_COLS
              and c not in ("_target", "__row_id__")]
if unassigned:
    print(f"NOTE: present but unused by the baseline: {unassigned}")

# Frequency tables are counted over the FULL training split, at its natural
# prevalence -- NOT over data["train"]["raw"].
#
# data["train"]["raw"] is train_balanced: every positive kept, negatives sampled
# down to ~5%. Counting there weights a positive row ~18x a negative one, and in
# the long tail (many distinct counterparty accounts, most seen a
# handful of times) that decides the feature: an account seen once in a positive
# scores roughly triple an otherwise identical account seen only in negatives.
# XGBoost learns "high logfreq -> positive" from the sampling design, and the
# relationship is absent -- or reversed -- at the natural prevalence of val and
# test. That drives test AUC *below* chance rather than merely to it.
#
# Counted one column at a time through .to_pandas().astype(str), the same code
# path raw_matrix() uses for the lookup, so the two string forms cannot diverge.
FULL_TRAIN = SPLIT_DIR / "train.parquet"
assert FULL_TRAIN.exists(), (
    f"Missing {FULL_TRAIN}  (notebook 01 writes it). The balanced sample is not a "
    "substitute here -- see the note above."
)

freq_maps = {}
for c in HIGH_CARD:
    _col = cudf.read_parquet(str(FULL_TRAIN), columns=[c])[c].to_pandas()
    freq_maps[c] = _col.astype(str).value_counts()
    print(f"  {c:<24} {len(freq_maps[c]):>9,} distinct over {len(_col):,} train rows")
    del _col


def raw_matrix(frame):
    out = pd.DataFrame(index=frame.index)
    for c in NUM_COLS:
        out[c] = pd.to_numeric(frame[c], errors="coerce").astype("float32")
    for c in CAT_COLS:
        out[c] = frame[c].astype(str)
    for c in HIGH_CARD:
        seen = frame[c].astype(str).map(freq_maps[c])
        # log1p(0) for a value the training period never saw: the honest
        # extrapolation, since unseen means rarest. There is deliberately no
        # separate __unseen flag -- every row of every split's feature frame is
        # covered by the train-split table or it is not, and the coverage check
        # below reports that directly instead of smuggling it in as a column
        # that is constant on the training matrix.
        out[f"{c}__logfreq"] = np.log1p(seen.fillna(0.0)).astype("float32")
    return out


# Coverage of the train-split frequency table, per split. Train must be ~100%:
# train_balanced is a subset of train, so anything below that means the two
# parquets disagree on the string form of an id column, not that a value is new.
print("\nFrequency-table coverage (share of rows whose value was seen in train):")
for split in ("train", "val", "test"):
    frame = data[split]["raw"]
    cov = {c: 1.0 - frame[c].astype(str).map(freq_maps[c]).isna().mean()
           for c in HIGH_CARD}
    print(f"  {split:<6} " + "  ".join(f"{c}={v:.2%}" for c, v in cov.items()))
    if split == "train":
        assert min(cov.values()) > 0.999, (
            f"train coverage {cov} -- the balanced sample contains values absent "
            f"from {FULL_TRAIN.name}. The id columns are being stringified "
            "differently in the two parquets; fix that before trusting anything."
        )

# Columns listed explicitly rather than selected by dtype (see the note above).
preprocessor = make_column_transformer(
    (OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_COLS),
    remainder="passthrough",
)

_train_mat = raw_matrix(data["train"]["raw"])
X_train_raw = preprocessor.fit_transform(_train_mat).astype(np.float32)
X_val_raw = preprocessor.transform(raw_matrix(data["val"]["raw"])).astype(np.float32)
X_test_raw = preprocessor.transform(raw_matrix(data["test"]["raw"])).astype(np.float32)

# ColumnTransformer emits the transformed block first, then the remainder in
# original column order.
RAW_COLS = CAT_COLS + [c for c in _train_mat.columns if c not in CAT_COLS]
assert len(RAW_COLS) == X_train_raw.shape[1], (
    f"{len(RAW_COLS)} names vs {X_train_raw.shape[1]} columns"
)

y_train, y_val, y_test = data["train"]["y"], data["val"]["y"], data["test"]["y"]
n_raw = X_train_raw.shape[1]

print(f"\nRaw feature matrix: {n_raw}d")
print(f"  dropped        : {DROP_COLS}")
print(f"  log-frequency  : {HIGH_CARD} -> {len(HIGH_CARD)} columns")
print(f"  ordinal        : {CAT_COLS}")
print(f"  numeric        : {NUM_COLS}")
print(f"  Train {X_train_raw.shape} | Val {X_val_raw.shape} | Test {X_test_raw.shape}")
for split, y in (("train", y_train), ("val", y_val), ("test", y_test)):
    print(f"  {split:<6} prevalence {y.mean():.4%}  ({int(y.sum()):,} positives)")

## 4. Train and Compare

**One hyperparameter set for all three arms.** The upstream notebook gave each arm its own
hand-tuned config -- the combined arm carried `gamma=4.8` and `min_child_weight=25.85`
against the raw arm's `gamma=0` and `12`. That varies the feature set *and* the
regularisation at once, so a difference cannot be attributed to either. An ablation over
feature sets needs everything else held fixed.

Two further changes:

- **Early stopping on `aucpr`, not `auc`.** Average precision is the headline metric, and
  NB01's conclusion notes AUROC saturates at low prevalence while AP still moves.
  Selecting iterations on AUC while judging on AP is a mismatch.
- **Train metrics reported alongside val/test, and AP expressed as lift over
  prevalence.** A test AP of 0.008 sounds like a number; against a 0.75% base rate it is a
  lift of 1.1x, i.e. chance. Lift is the figure to read, and the train/test gap is what
  exposes memorisation.

In [ ]:
# One config for every arm -- see the note above.
XGB_PARAMS = {
    "n_estimators": 400, "max_depth": 8, "learning_rate": 0.05,
    "colsample_bytree": 0.95, "min_child_weight": 12, "subsample": 0.673,
    "reg_alpha": 0.01, "reg_lambda": 0.001, "random_state": RANDOM_STATE,
}


if EXP is not None:
    XGB_PARAMS.update(EXP.xgboost_params())

from src.evaluation import head_metrics, ranking_metrics, save_predictions

PREDICTIONS = {}   # arm display name -> {split: scores}


def train_xgb(X_tr, X_va, X_te, name):
    print(f"\nTraining {name}...")
    print(f"  Features: {X_tr.shape[1]}d | Samples: {X_tr.shape[0]:,}")
    t0 = time.time()
    clf = xgb.XGBClassifier(
        **XGB_PARAMS,
        scale_pos_weight=1.0,
        tree_method="hist",
        device=XGB_DEVICE,
        early_stopping_rounds=20,
        eval_metric="aucpr",
    )
    clf.fit(X_tr, y_train, eval_set=[(X_va, y_val)], verbose=False)

    preds = {"train": clf.predict_proba(X_tr)[:, 1],
             "val": clf.predict_proba(X_va)[:, 1],
             "test": clf.predict_proba(X_te)[:, 1]}
    truth = {"train": y_train, "val": y_val, "test": y_test}

    metrics = {"n_features": int(X_tr.shape[1]),
               "best_iteration": int(clf.best_iteration)}
    # src/evaluation.py: ROC-AUC (+ standard error), AP, lift, recall at fixed
    # false-positive rates, precision in the top scores -- per split.
    for split in ("train", "val", "test"):
        for key, value in ranking_metrics(truth[split], preds[split]).items():
            metrics[f"{split}_{key}"] = value
    PREDICTIONS[name] = preds          # saved with the run, see the results cell

    print(f"  Train time: {time.time() - t0:.1f}s "
          f"(best_iteration={metrics['best_iteration']} of {XGB_PARAMS['n_estimators']})")
    print(f"  {'split':<7}{'ROC-AUC':>9}{'AP':>10}{'AP/prev':>9}")
    for split in ("train", "val", "test"):
        print(f"  {split:<7}{metrics[f'{split}_auc']:>9.4f}"
              f"{metrics[f'{split}_ap']:>10.4f}{metrics[f'{split}_lift']:>8.2f}x")
    if metrics["test_lift"] < 1.5 and metrics["train_lift"] - metrics["test_lift"] > 2.0:
        print("  !! fits train but not test -- memorisation, not signal")
    return clf, metrics


print("=" * 66)
print("Training XGBoost models")
print("=" * 66)

clf_raw, m_raw = train_xgb(X_train_raw, X_val_raw, X_test_raw,
                           f"Raw features ({n_raw}d)")
clf_embed, m_embed = train_xgb(X_train_pca, X_val_pca, X_test_pca, EMB_LABEL)

# float32: X_*_raw is already float32, but hstack with a float64 embedding array
# would promote the whole 1M x (n_raw + 512) matrix to 8 bytes per cell.
X_train_comb = np.hstack([X_train_raw, X_train_pca]).astype(np.float32)
X_val_comb = np.hstack([X_val_raw, X_val_pca]).astype(np.float32)
X_test_comb = np.hstack([X_test_raw, X_test_pca]).astype(np.float32)
clf_comb, m_comb = train_xgb(X_train_comb, X_val_comb, X_test_comb,
                             f"Combined ({n_raw}+{X_train_pca.shape[1]}d)")

## 5. Results

In [ ]:
# Logistic regression on the same three feature sets: a sanity check next to
# XGBoost (src/downstream.py) -- same rows, same metrics, C chosen on val.
#   logreg ~ XGBoost on embeddings -> the signal is linearly readable from them
#   XGBoost >> logreg on raw only  -> trees find interactions the embedding hides
#   logreg > XGBoost anywhere      -> XGBoost overfits or is mistuned there
LOGREG_METRICS, LOGREG_SCORES = {}, {}
if LOGREG:
    from src.downstream import fit_logreg

    for _key, (_tr, _va, _te) in {
        "raw": (X_train_raw, X_val_raw, X_test_raw),
        "embeddings": (X_train_pca, X_val_pca, X_test_pca),
        "combined": (X_train_comb, X_val_comb, X_test_comb),
    }.items():
        _t0 = time.time()
        _, _scores, _m = fit_logreg(_tr, y_train, _va, y_val, _te, y_test, seed=RANDOM_STATE)
        LOGREG_METRICS[f"logreg_{_key}"] = _m
        LOGREG_SCORES[f"logreg_{_key}"] = _scores
        print(f"Logistic regression, {_key:<10}: test ROC-AUC {_m['test_auc']:.4f}  "
              f"AP {_m['test_ap']:.4f}  (C={_m['C']}, {time.time() - _t0:.0f}s)"
              + ("" if _m["converged"] else "  !! did not converge (max_iter reached)"))

In [ ]:
results = {
    f"Raw features ({n_raw}d)": m_raw,
    EMB_LABEL: m_embed,
    f"Combined ({n_raw}+{X_train_pca.shape[1]}d)": m_comb,
}

names = list(results)
aucs = [results[n]["test_auc"] for n in names]
aps = [results[n]["test_ap"] for n in names]
best_auc = names[int(np.argmax(aucs))]
best_ap = names[int(np.argmax(aps))]
prevalence = float(y_test.mean())

# Machine-readable results (scripts/run_experiment.py collects them into
# summary.csv) and the fitted models, so a run can be re-scored without retraining.
MODELS_DIR = OUTPUTS_DIR / "xgb_models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
for _key, _clf in (("raw", clf_raw), ("embeddings", clf_embed), ("combined", clf_comb)):
    _clf.save_model(str(MODELS_DIR / f"{_key}.json"))
METRICS = {
    "arms": {"raw": m_raw, "embeddings": m_embed, "combined": m_comb, **LOGREG_METRICS},
    "n_test_positives": int(y_test.sum()),
    "prevalence_test": prevalence,
    "n_train_rows": int(len(y_train)),
    "embedding": {k: metadata.get(k) for k in
                  ("pooling", "history_window", "learned_head", "tokens_per_transaction")},
    "xgb_params": XGB_PARAMS,
}
# Learned pooling: the attention head also scores on its own. Its train scores
# cover every train row, so they get the same head-row filter as data["train"].
HEAD_SCORES = {}
if metadata.get("head_rows_file"):
    for _s in ("train", "val", "test"):
        _p = EMBED_DIR / f"{_s}_head_scores.npy"
        if _p.exists():
            _v = np.load(_p)
            HEAD_SCORES[_s] = _v[~head_rows] if _s == "train" else _v
if HEAD_SCORES:
    METRICS["head"] = head_metrics(HEAD_SCORES, {s: data[s]["y"] for s in HEAD_SCORES})
    METRICS["head_test_auc"] = METRICS["head"]["test"]["auc"]

# Every score of every model, per split, with the row id and label: any other
# metric can be computed later from these files (src/evaluation.py,
# metrics_from_predictions) without retraining anything.
ARM_KEYS = dict(zip(names, ("raw", "embeddings", "combined")))
PRED_FILES = save_predictions(
    OUTPUTS_DIR / "predictions",
    {s: {"__row_id__": data[s]["raw"]["__row_id__"].to_numpy(), "label": data[s]["y"]}
     for s in ("train", "val", "test")},
    {**{ARM_KEYS[n]: PREDICTIONS[n] for n in names}, **LOGREG_SCORES,
     **({"head": HEAD_SCORES} if HEAD_SCORES else {})},
)
print(f"Predictions -> {OUTPUTS_DIR / 'predictions'} ({', '.join(PRED_FILES)})")

# Fraud value caught at fixed alert budgets (shares only, src/evaluation.py),
# per arm, on val and test. Optional: skipped if the amount column is missing.
try:
    from src.evaluation import value_metrics

    _amount_col = SCH.get("amount", "amt_euro")
    _all_scores = {**{ARM_KEYS[n]: PREDICTIONS[n] for n in names}, **LOGREG_SCORES}
    _value = {}                       # merged only once every arm succeeded
    for _arm, _by_split in _all_scores.items():
        for _s in ("val", "test"):
            _amt = pd.to_numeric(data[_s]["raw"][_amount_col], errors="coerce").to_numpy()
            for _k, _v in value_metrics(data[_s]["y"], _by_split[_s], _amt).items():
                _value.setdefault(_arm, {})[f"{_s}_{_k}"] = _v
    for _arm, _vals in _value.items():
        METRICS["arms"][_arm].update(_vals)
    print(f"{'Model':<24}{'value caught, top 1%':>22}{'frauds caught, top 1%':>23}")
    for _arm, _m in METRICS["arms"].items():
        if "test_value_recall_at_top_1pct" in _m:
            print(f"{_arm:<24}{_m['test_value_recall_at_top_1pct']:>21.1%}"
                  f"{_m['test_recall_at_top_1pct']:>22.1%}")
except Exception as exc:  # noqa: BLE001
    print(f"(value metrics skipped: {exc!r})")

(OUTPUTS_DIR / "metrics.json").write_text(json.dumps(METRICS, indent=2, default=float))
print(f"Metrics -> {OUTPUTS_DIR / 'metrics.json'}\n")

# Drift: test metrics per month for every model (src/evaluation.py). Optional:
# it runs after metrics.json is written and cannot fail the run.
try:
    from src.evaluation import metrics_by_period

    BY_MONTH = metrics_by_period(
        data["test"]["y"],
        {**{ARM_KEYS[n]: PREDICTIONS[n]["test"] for n in names},
         **{k: v["test"] for k, v in LOGREG_SCORES.items()}},
        data["test"]["raw"][SCH.get("datetime", "datetime")],
        date_format=TOK_CFG.get("datetime_format"),
    )
    BY_MONTH.to_csv(OUTPUTS_DIR / "metrics_by_month.csv", index=False)
    print("Test ROC-AUC per month:")
    print(BY_MONTH.pivot(index="period", columns="model", values="auc").round(3).to_string())
except Exception as exc:  # noqa: BLE001
    print(f"(per-month metrics skipped: {exc!r}; scripts/analyze_predictions.py drift can redo them)")

print("=" * 90)
print(f"RESULTS (test set -- final holdout, prevalence {prevalence:.4%})")
print("=" * 90)
print(f"{'Model':<32}{'Feat':>6}{'ROC-AUC':>9}{'AP':>9}{'AP/prev':>9}"
      f"{'train AP/prev':>15}{'Best':>8}")
print("-" * 90)
for name in names:
    m = results[name]
    badge = ",".join(
        b for b, c in (("AUC", name == best_auc), ("AP", name == best_ap)) if c
    )
    print(f"{name:<32}{m['n_features']:>6}{m['test_auc']:>9.4f}{m['test_ap']:>9.4f}"
          f"{m['test_lift']:>8.2f}x{m['train_lift']:>14.2f}x{badge:>8}")

print("\n" + "=" * 90)
print("LIFT OVER THE RAW-FEATURE BASELINE (test set)")
print("=" * 90)
base = results[names[0]]
for name in names[1:]:
    m = results[name]
    d_auc = (m["test_auc"] - base["test_auc"]) / base["test_auc"] * 100
    d_ap = (m["test_ap"] - base["test_ap"]) / max(base["test_ap"], 1e-12) * 100
    print(f"  {name:<36} ROC-AUC {d_auc:+7.2f}%   AP {d_ap:+7.2f}%")

# A model at chance ranks no better than the base rate. Say so, rather than
# letting a small AP read as a small success.
weak = [n for n in names if results[n]["test_lift"] < 1.5]
if weak:
    print("\n  !! at or near chance (AP/prevalence < 1.5x): " + "; ".join(weak))
    print("     Differences between arms at this level are sampling noise, not")
    print("     evidence about the embeddings. Verify the label before tuning models.")

nb01 = OUTPUTS_DIR / "baseline_metrics.json"
if nb01.exists():
    nb01_metrics = json.loads(nb01.read_text())["metrics"]
    print(f"\nNotebook 01 baseline: test AUROC {nb01_metrics['test_auc']:.4f}, "
          f"test AP {nb01_metrics['test_ap']:.4f}")
    print("  (NB01 still uses the upstream encoding, including customer_id and _ts, so a")
    print("   higher-looking number there reflects memorisation, not a better model.)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
n_emb = X_train_pca.shape[1]
short = [f"Raw\n({n_raw}d)", f"Embeddings\n({n_emb}d)", f"Combined\n({n_raw}+{n_emb}d)"]
colors = ["#1f77b4", "#2ca02c", "#ff7f0e"]

for ax, values, title, ylabel in (
    (ax1, aucs, "ROC-AUC (overall ranking)", "ROC-AUC"),
    (ax2, aps, "Average Precision (flagging quality)", "Average precision"),
):
    bars = ax.bar(short, values, color=colors, edgecolor="black", linewidth=1)
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold",
        )
    bars[int(np.argmax(values))].set_edgecolor("red")
    bars[int(np.argmax(values))].set_linewidth(3)
    ax.axhline(values[0], color="#1f77b4", linestyle="--", alpha=0.5, linewidth=1.5)
    # Chance level: AP equal to the base rate means no ranking ability at all.
    # Without this line a bar at 0.008 looks like a result rather than noise.
    if ylabel.startswith("Average"):
        ax.axhline(prevalence, color="crimson", linestyle=":", linewidth=1.6,
                   label=f"prevalence = {prevalence:.4f} (chance)")
        ax.legend(fontsize=9, loc="upper left")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(axis="y", alpha=0.3)

ax1.set_ylim(min(aucs) * 0.95, max(aucs) * 1.02)
ax2.set_ylim(0, max(max(aps), prevalence) * 1.35)
fig.suptitle(
    f"Detection performance (test set, history window = {metadata['history_window']} txns)",
    fontsize=15, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "xgb_auc_ap_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {OUTPUTS_DIR / 'xgb_auc_ap_comparison.png'}")

## 6. Confusion Matrices at a Fixed Alert Budget

The bar chart above compares arms on ranking metrics. This section turns the same
predictions into the counts a reviewer sees.

There is no useful confusion matrix at the usual 0.5 threshold: each arm is trained
on a ~1%-positive sample with `scale_pos_weight=1`, so almost nothing scores above
0.5 and all three degenerate to "predict negative". The operating point here is an
**alert budget** instead -- flag the top B% of test transactions by score -- which
depends only on the ranking and is what a review team actually chooses.

Compare every cell against the chance reference in the panel title: flagging B% of
rows at random catches B% of the positives.

In [ ]:
# A confusion matrix at the default 0.5 threshold would say nothing here. Each arm
# is trained on a ~1%-positive sample with scale_pos_weight=1, so almost no score
# reaches 0.5 and all three collapse to the same "predict negative" matrix --
# [[n_negatives, 0], [n_positives, 0]] -- regardless of how well they rank.
#
# The operating point below is a review budget instead: flag the top B% of test
# transactions by score. That is a pure function of the ranking (the thing AUROC
# and AP measure), it is comparable across arms, and it is the decision a fraud
# team actually makes -- how many alerts can be investigated.
#
# Read every cell against the chance reference: flagging B% of rows at random
# catches B% of the positives. An arm that matches that number has no ranking
# ability, whatever the matrix looks like. Inverting its scores would not help --
# AUROC is symmetric about 0.5, so an arm just under 0.5 inverted is an arm just over it, still inside
# the noise band, with the sign chosen by peeking at the test set.

ALERT_BUDGETS = (0.001, 0.01, 0.05)   # share of test transactions reviewed
BUDGET_SHOWN = 0.01                   # the budget drawn as matrices below

test_scores = {
    f"Raw features ({n_raw}d)": clf_raw.predict_proba(X_test_raw)[:, 1],
    EMB_LABEL: clf_embed.predict_proba(X_test_pca)[:, 1],
    f"Combined ({n_raw}+{X_train_pca.shape[1]}d)": clf_comb.predict_proba(X_test_comb)[:, 1],
}

n_test = len(y_test)
n_pos = int(y_test.sum())
prev = n_pos / n_test
_rng = np.random.default_rng(RANDOM_STATE)
_tiebreak = _rng.random(n_test)


def confusion_at_budget(scores, budget):
    """Flag the top *budget* share by score; return (tn, fp, fn, tp).

    Ties are broken at random rather than by row order. That matters precisely in
    the degenerate case being diagnosed: a model that outputs a near-constant
    score would otherwise have its "top k" decided by the parquet's row order,
    which is sorted by customer -- turning an artefact into an apparent result.
    """
    k = max(1, int(round(budget * n_test)))
    order = np.lexsort((_tiebreak, -np.asarray(scores, dtype=np.float64)))
    flagged = np.zeros(n_test, dtype=bool)
    flagged[order[:k]] = True
    tp = int((flagged & (y_test == 1)).sum())
    fp = int(flagged.sum() - tp)
    return n_test - n_pos - fp, fp, n_pos - tp, tp


print(f"Test set: {n_test:,} transactions, {n_pos:,} positives ({prev:.3%})")
for budget in ALERT_BUDGETS:
    k = max(1, int(round(budget * n_test)))
    print(f"\nAlert budget {budget:.1%}  --  {k:,} transactions reviewed, "
          f"a random ranking catches {budget * n_pos:.1f} of {n_pos:,}")
    print(f"  {'model':<34}{'TP':>6}{'FP':>9}{'recall':>9}{'precision':>11}{'lift':>8}")
    for name, scores in test_scores.items():
        _, fp, _, tp = confusion_at_budget(scores, budget)
        recall, precision = tp / max(n_pos, 1), tp / max(tp + fp, 1)
        print(f"  {name:<34}{tp:>6}{fp:>9,}{recall:>8.1%}"
              f"{precision:>11.3%}{precision / prev:>7.2f}x")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
k_shown = max(1, int(round(BUDGET_SHOWN * n_test)))
chance_tp = BUDGET_SHOWN * n_pos
# Under a random ranking TP is Binomial(k, prevalence), so its spread is about
# sqrt(chance_tp) -- at a 1% budget that is 7.5 +/- 2.7, and a flat "1.5x chance"
# rule would call a coin flip a result about one run in five. Three sigma is the
# bar an arm has to clear before it counts as ranking at all.
chance_sd = np.sqrt(chance_tp)
beats_chance = chance_tp + 3 * chance_sd
at_chance = []

for ax, (name, scores) in zip(axes, test_scores.items()):
    tn, fp, fn, tp = confusion_at_budget(scores, BUDGET_SHOWN)
    cm = np.array([[tn, fp], [fn, tp]], dtype=float)
    # Row-normalised colour, raw counts annotated: the classes differ by two
    # orders of magnitude, so colouring by count would render one row invisible.
    norm = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    for (r, c), value in np.ndenumerate(cm):
        ax.text(c, r, f"{int(value):,}\n{norm[r, c]:.1%}", ha="center", va="center",
                fontsize=11, fontweight="bold",
                color="white" if norm[r, c] > 0.5 else "black")
    ax.set_xticks([0, 1], ["Not flagged", "Flagged"])
    ax.set_yticks([0, 1], ["Normal", "Positive"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    weak = tp < beats_chance
    if weak:
        at_chance.append(name)
    ax.set_title(
        f"{name}\nTP {tp} vs {chance_tp:.1f} +/- {chance_sd:.1f} at chance",
        fontsize=11, fontweight="bold", color="crimson" if weak else "black",
    )

fig.suptitle(
    f"Confusion at a {BUDGET_SHOWN:.1%} alert budget "
    f"(top {k_shown:,} of {n_test:,} test transactions by score)",
    fontsize=14, fontweight="bold", y=1.04,
)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "xgb_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved -> {OUTPUTS_DIR / 'xgb_confusion_matrices.png'}")

if at_chance:
    print(f"\n  !! within 3 sigma of chance at this budget (would need TP > "
          f"{beats_chance:.0f}): " + "; ".join(at_chance))
    print("     The matrices differ between arms only by sampling noise. Inverting")
    print("     a below-chance arm does not recover signal -- it moves AUROC the")
    print("     same distance back across 0.5 and leaves it at chance. Establish")
    print("     that the label is stationary (scripts/diagnose_label.py) before")
    print("     reading anything into these numbers.")

## 7. How to Read the Outcome

Interpret the three numbers together, not the headline lift alone:

- **Combined > raw, embeddings < raw.** The expected shape for a decoder-only model.
  Embeddings contribute sequential context (rhythm, counterparty history, timing) that
  raw fields cannot express, but they lose fine-grained per-field precision. Their value
  is as a complement.
- **Combined == raw.** The embeddings add nothing yet. In order of likelihood: the
  model is undertrained (notebook 03 demo runs 30 steps -- try a few thousand),
  `HISTORY_WINDOW` is too small for the behaviour you care about, or the vocabulary caps
  collapse the discriminative fields into `*_OTHER` (check the fitted sizes printed in
  notebook 02).
- **Embeddings alone > raw.** Verify before celebrating: it usually means label leakage
  through a field that is only populated after a case is opened, or `operations_status`
  encoding the outcome.

Things worth trying next, roughly by expected payoff:

1. **Pretrain longer** on the full train corpus, watching validation loss.
2. **Sweep `HISTORY_WINDOW`** (8 / 32 / 128) -- it directly trades context for compute
   and is the biggest single lever on embedding quality.
3. **Tune the vocabulary caps** in notebook 02 for the fields your analysts consider
   most informative (`movement_code`, `counterparty_bic`).
4. **Try mean pooling** (`pooling="mean"` in notebook 04) as a comparison to last-token.
5. **Run HPO** on all three models before quoting a lift number.

**Outputs:** `outputs/model_comparison.json`, `outputs/xgb_auc_ap_comparison.png`